# Table 1 Cohort Descriptors: Initial Rhythm and Arrest Location

This notebook generates descriptive rows for manuscript Table 1:

- non-shockable rhythm percentage for each cohort when source data are available;
- MIMIC-IV rhythm from the initial-rhythm feature file created by `mimiciv/identify_initial_rhythm.ipynb`;
- OHCA/IHCA or ED-presenting/procedure-coded composition for each cohort.

The notebook intentionally uses conservative labels. For eICU and PMAP, location is inferred from admission/ED fields and should be described as an ED-presenting/OHCA proxy unless manually validated against source documentation. For HYPERION, `J0_LIEU_ACR` is used to classify home/outside as OHCA and hospital as IHCA. For MIMIC-IV, `CA_ED.csv` and `CA_time.csv` are used when available to distinguish ED-presenting cardiac arrest from procedure-coded CPR events.

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "eICU").exists() and (ROOT.parent / "eICU").exists():
    ROOT = ROOT.parent

OUT = ROOT / "summarized_results" / "table1_data_pulls"
OUT.mkdir(parents=True, exist_ok=True)

# These are the Table 1 denominators in the current manuscript table.
# The notebook refuses silent raw-cohort denominators unless explicitly allowed.
EXPECTED_TABLE1_N = {
    "HYPERION": 581,
    "PMAP": 1292,
    "eICU": 2466,
    "MIMIC-IV": 552,
}
ALLOW_N_MISMATCH = os.environ.get("ALLOW_TABLE1_N_MISMATCH", "0").strip().lower() in {"1", "true", "yes"}

PATHS = {
    "HYPERION": [
        ROOT / "hyperion" / "predictorsDf.csv",
    ],
    "eICU": [
        ROOT / "eICU" / "eICUPredictorsDiag.csv",
        ROOT / "eICU" / "eICUPredictors.csv",
        ROOT / "eICU" / "eICUPredictors1.csv",
    ],
    "PMAP": [
        ROOT / "pmap" / "PMAP_Predictors.csv",
        ROOT / "pmap" / "PMAP_Predictors2.csv",
    ],
    "MIMIC-IV": [
        ROOT / "mimiciv" / "MIMIC_Predictors.csv",
        ROOT / "mimiciv" / "MIMIC_Predictors1.csv",
        ROOT / "mimiciv" / "MIMIC_Predictors2.csv",
    ],
}

MIMIC_CA_ED = Path(os.environ.get("MIMIC_CA_ED_CSV", str(ROOT / "mimiciv" / "CA_ED.csv")))
MIMIC_CA_TIME = Path(os.environ.get("MIMIC_CA_TIME_CSV", str(ROOT / "mimiciv" / "CA_time.csv")))
MIMIC_RHYTHM_FEATURES = Path(os.environ.get("MIMIC_INITIAL_RHYTHM_CSV", str(ROOT / "mimiciv" / "MIMIC_initial_rhythm_features.csv")))

print("Repository root:", ROOT)
print("Output directory:", OUT)
print("Expected Table 1 n:", EXPECTED_TABLE1_N)

In [ ]:
UNSCORABLE = "Unable to score due to medication"


def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


def require_columns(df, cols, label):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{label}: missing required columns: {missing}")


def apply_eicu_dml_filter(df, outcome="mortality"):
    out = df.copy()
    require_columns(
        out,
        ["LastMGCS", "FirstMGCSTime", "LastMGCSTime", "DeathAtDischarge", "nurse_first_Motor", "Hypothermia"],
        "eICU DML filter",
    )
    f = (out["LastMGCS"] != UNSCORABLE) & (~out["LastMGCS"].isna())
    f = f & (out["FirstMGCSTime"] != out["LastMGCSTime"])
    for c in ["FirstGCS", "FirstMGCS", "LastMGCS", "LastGCS"]:
        if c in out.columns:
            out.loc[out[c] == UNSCORABLE, c] = np.nan
    out.loc[out["DeathAtDischarge"] == 1, "LastMGCS"] = 1
    out.loc[f, "LastMGCSPositive"] = (out.loc[f, "LastMGCS"].astype(float) == 6).astype(int)

    gcs15_filter = out["nurse_first_Motor"] != 6
    treatment_known = ~out["Hypothermia"].isna()
    if outcome == "neuro":
        mask = f & gcs15_filter & treatment_known
    elif outcome == "mortality":
        mask = f & gcs15_filter & ~out["DeathAtDischarge"].isna() & treatment_known
    else:
        raise ValueError(f"Unknown eICU outcome filter: {outcome}")
    return out.loc[mask].copy()


def apply_epic_dml_filter(df, dataset, outcome="mortality"):
    out = df.copy()
    require_columns(
        out,
        ["first_mGCS", "first_mGCS_time", "last_mGCS_time", "last_mGCS", "death_at_disch", "hypothermia"],
        f"{dataset} DML filter",
    )
    time_changed = out["first_mGCS_time"] != out["last_mGCS_time"]
    out.loc[out["death_at_disch"] == 1, "last_mGCS"] = 1
    out.loc[time_changed, "LastMGCSPositive"] = (
        out.loc[time_changed, "last_mGCS"].astype(float) == 6
    ).astype(int)

    gcs15_filter = out["first_mGCS"] != 6
    treatment_known = ~out["hypothermia"].isna()
    if outcome == "neuro":
        mask = gcs15_filter & time_changed & treatment_known
    elif outcome == "mortality":
        # PMAP/MIMIC DML utilities do not apply the follow-up motor-GCS time filter for mortality.
        mask = gcs15_filter & ~out["death_at_disch"].isna() & treatment_known
    else:
        raise ValueError(f"Unknown {dataset} outcome filter: {outcome}")
    return out.loc[mask].copy()


def apply_hyperion_dml_filter(df, outcome="mortality"):
    out = df.copy()
    treatment_col = find_col(out, ["groupe", "TTM", "hypothermia"])
    if treatment_col is None:
        raise ValueError("HYPERION DML filter: missing treatment column groupe/TTM/hypothermia")
    if outcome == "mortality":
        outcome_col = find_col(out, ["hospital_mortality", "DeathAtDischarge", "death_at_disch"])
    elif outcome == "neuro":
        outcome_col = find_col(out, ["LastMGCSPositive", "CPC12", "CPC_SC3"])
    else:
        raise ValueError(f"Unknown HYPERION outcome filter: {outcome}")
    if outcome_col is None:
        raise ValueError(f"HYPERION DML filter: missing outcome column for {outcome}")

    treatment = pd.to_numeric(out[treatment_col], errors="coerce")
    mask = treatment.ne(2) & treatment.notna() & ~out[outcome_col].isna()
    return out.loc[mask].copy()


def load_csv(label, paths, expected_n=None, postprocess=None):
    found = []
    for candidate in paths:
        if Path(candidate).exists():
            raw = pd.read_csv(candidate)
            df = postprocess(raw) if postprocess else raw
            found.append((Path(candidate), raw, df))

    if not found:
        print(f"{label}: no predictor CSV found in candidates:")
        for candidate in paths:
            print("  -", candidate)
        return pd.DataFrame(), None

    print(f"{label}: candidate predictor CSVs")
    for path, raw, df in found:
        marker = ""
        if expected_n is not None and len(df) == expected_n:
            marker = " <-- matches Table 1 n"
        print(f"  - {path}: raw_shape={raw.shape}; analysis_shape={df.shape}{marker}")

    selected = None
    if expected_n is not None:
        for path, raw, df in found:
            if len(df) == expected_n:
                selected = (path, df)
                break

    if selected is None:
        path, raw, df = found[0]
        selected = (path, df)

    path, df = selected
    if expected_n is not None and len(df) != expected_n:
        msg = (
            f"{label}: selected {path.name} has n={len(df)}, but Table 1 expects n={expected_n}. "
            "Do not use these rows for Table 1 until the final cohort CSV is present or the expected n is updated."
        )
        if ALLOW_N_MISMATCH:
            print("WARNING:", msg)
        else:
            raise ValueError(msg)

    print(f"{label}: selected {path} with analysis shape {df.shape}")
    return df, path


def find_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def boolish(series):
    if series is None:
        return pd.Series(dtype=float)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0) != 0
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["1", "true", "t", "yes", "y", "present", "positive"])


def summarize_binary(dataset, variable, values, source, denominator_note="analysis cohort"):
    values = pd.Series(values)
    denom = int(values.notna().sum())
    num = int(values.fillna(False).astype(bool).sum()) if denom else 0
    pct = 100 * num / denom if denom else np.nan
    return {
        "dataset": dataset,
        "variable": variable,
        "n": num,
        "denominator": denom,
        "percent": pct,
        "formatted": "NR" if denom == 0 else f"{num}/{denom} ({pct:.1f}%)",
        "source": source,
        "denominator_note": denominator_note,
    }


def merge_by_available_id(left, right):
    id_priority = [
        ["stay_id"],
        ["hadm_id"],
        ["subject_id"],
        ["osler_id"],
        ["pat_enc_csn_id"],
        ["patientunitstayid"],
    ]
    for keys in id_priority:
        if all(k in left.columns and k in right.columns for k in keys):
            l = left.copy()
            r = right.copy()
            for k in keys:
                l[k] = pd.to_numeric(l[k], errors="ignore")
                r[k] = pd.to_numeric(r[k], errors="ignore")
            return l.merge(r.drop_duplicates(keys), on=keys, how="left"), keys
    return left.copy(), []

## Load Predictor Cohorts

In [ ]:
hyperion, hyperion_path = load_csv(
    "HYPERION",
    PATHS["HYPERION"],
    EXPECTED_TABLE1_N["HYPERION"],
    postprocess=lambda df: apply_hyperion_dml_filter(df, outcome="mortality"),
)
eicu, eicu_path = load_csv(
    "eICU",
    PATHS["eICU"],
    EXPECTED_TABLE1_N["eICU"],
    postprocess=lambda df: apply_eicu_dml_filter(df, outcome="mortality"),
)
pmap, pmap_path = load_csv(
    "PMAP",
    PATHS["PMAP"],
    EXPECTED_TABLE1_N["PMAP"],
    postprocess=lambda df: apply_epic_dml_filter(df, "PMAP", outcome="mortality"),
)
mimic, mimic_path = load_csv(
    "MIMIC-IV",
    PATHS["MIMIC-IV"],
    EXPECTED_TABLE1_N["MIMIC-IV"],
    postprocess=lambda df: apply_epic_dml_filter(df, "MIMIC-IV", outcome="mortality"),
)

if hyperion_path is not None and not hyperion.empty:
    hyperion_raw = pd.read_csv(hyperion_path)
    hyperion_neuro = apply_hyperion_dml_filter(hyperion_raw, outcome="neuro")
    print(f"HYPERION DML neuro analysis shape for audit only: {hyperion_neuro.shape}")
if eicu_path is not None and not eicu.empty:
    eicu_raw = pd.read_csv(eicu_path)
    eicu_neuro = apply_eicu_dml_filter(eicu_raw, outcome="neuro")
    print(f"eICU DML neuro analysis shape for audit only: {eicu_neuro.shape}")
if pmap_path is not None and not pmap.empty:
    pmap_raw = pd.read_csv(pmap_path)
    pmap_neuro = apply_epic_dml_filter(pmap_raw, "PMAP", outcome="neuro")
    print(f"PMAP DML neuro analysis shape for audit only: {pmap_neuro.shape}")
if mimic_path is not None and not mimic.empty:
    mimic_raw = pd.read_csv(mimic_path)
    mimic_neuro = apply_epic_dml_filter(mimic_raw, "MIMIC-IV", outcome="neuro")
    print(f"MIMIC-IV DML neuro analysis shape for audit only: {mimic_neuro.shape}")

for label, df in [("HYPERION", hyperion), ("eICU", eicu), ("PMAP", pmap), ("MIMIC-IV", mimic)]:
    if not df.empty:
        print(f"\n{label} columns containing rhythm/location keywords:")
        cols = [c for c in df.columns if re.search(r"rhythm|rythm|vtach|vfib|asystole|pea|\bvf\b|shock|ed_visit|admit|source|origin|location|lieu|acr", c, re.I)]
        print(cols[:120])

## Non-Shockable Rhythm Row for Table 1

In [ ]:
rhythm_rows = []
rhythm_components = []

if not hyperion.empty:
    rhythm_col = find_col(hyperion, ["J0_RYTHM"])
    if rhythm_col:
        rhythm = pd.to_numeric(hyperion[rhythm_col], errors="coerce")
        known = rhythm.isin([1, 2])
        asys = rhythm.eq(1).where(known, np.nan)
        pea = rhythm.eq(2).where(known, np.nan)
        nonshock = rhythm.isin([1, 2]).where(known, np.nan)
        rhythm_rows.append(summarize_binary(
            "HYPERION",
            "Non-shockable initial rhythm (asystole/PEA)",
            nonshock,
            source=f"{hyperion_path}; {rhythm_col}: 1=asystole, 2=PEA",
            denominator_note="known J0_RYTHM",
        ))
        rhythm_components.extend([
            summarize_binary("HYPERION", "Asystole", asys, f"{hyperion_path}; {rhythm_col}=1"),
            summarize_binary("HYPERION", "PEA", pea, f"{hyperion_path}; {rhythm_col}=2"),
        ])
    else:
        rhythm_rows.append({
            "dataset": "HYPERION",
            "variable": "Non-shockable initial rhythm (asystole/PEA)",
            "n": np.nan,
            "denominator": len(hyperion),
            "percent": np.nan,
            "formatted": "NR",
            "source": f"{hyperion_path}; no J0_RYTHM column found.",
            "denominator_note": "not reported",
        })

if not eicu.empty:
    vt_col = find_col(eicu, ["VTachy", "diagnosis_initial rhythm: ventricular tachycardia", "diagnosis_ventricular tachycardia"])
    vf_col = find_col(eicu, ["VFib", "diagnosis_initial rhythm: ventricular fibrillation", "diagnosis_ventricular fibrillation"])
    asys_col = find_col(eicu, ["Asystole", "diagnosis_initial rhythm: asystole"])
    pea_col = find_col(eicu, ["PEA", "diagnosis_initial rhythm: pulseless electrical activity"])

    asys = boolish(eicu[asys_col]) if asys_col else pd.Series(False, index=eicu.index)
    pea = boolish(eicu[pea_col]) if pea_col else pd.Series(False, index=eicu.index)
    vt = boolish(eicu[vt_col]) if vt_col else pd.Series(False, index=eicu.index)
    vf = boolish(eicu[vf_col]) if vf_col else pd.Series(False, index=eicu.index)
    any_rhythm = asys | pea | vt | vf
    nonshock = asys | pea

    rhythm_rows.append(summarize_binary(
        "eICU-CRD",
        "Non-shockable initial rhythm (asystole/PEA)",
        nonshock,
        source=f"{eicu_path}; columns asystole={asys_col}, pea={pea_col}, vt={vt_col}, vf={vf_col}",
    ))
    rhythm_components.extend([
        summarize_binary("eICU-CRD", "Asystole", asys, f"{eicu_path}; {asys_col}"),
        summarize_binary("eICU-CRD", "PEA", pea, f"{eicu_path}; {pea_col}"),
        summarize_binary("eICU-CRD", "VT", vt, f"{eicu_path}; {vt_col}"),
        summarize_binary("eICU-CRD", "VF", vf, f"{eicu_path}; {vf_col}"),
        summarize_binary("eICU-CRD", "Any documented rhythm flag", any_rhythm, f"{eicu_path}"),
    ])

if not pmap.empty:
    asys_col = find_col(pmap, ["asystole", "cardiac asystole", "asystole by electrocardiogram"])
    pea_col = find_col(pmap, ["pea", "pulseless electrical activity", "cardiac arrest with pulseless electrical activity"])
    vf_col = find_col(pmap, ["VF", "ventricular fibrillation", "cardiac arrest with ventricular fibrillation"])

    asys = boolish(pmap[asys_col]) if asys_col else pd.Series(False, index=pmap.index)
    pea = boolish(pmap[pea_col]) if pea_col else pd.Series(False, index=pmap.index)
    vf = boolish(pmap[vf_col]) if vf_col else pd.Series(False, index=pmap.index)
    any_rhythm = asys | pea | vf
    nonshock = asys | pea

    rhythm_rows.append(summarize_binary(
        "PMAP",
        "Non-shockable initial rhythm (asystole/PEA)",
        nonshock,
        source=f"{pmap_path}; columns asystole={asys_col}, pea={pea_col}, vf={vf_col}",
    ))
    rhythm_components.extend([
        summarize_binary("PMAP", "Asystole", asys, f"{pmap_path}; {asys_col}"),
        summarize_binary("PMAP", "PEA", pea, f"{pmap_path}; {pea_col}"),
        summarize_binary("PMAP", "VF", vf, f"{pmap_path}; {vf_col}"),
        summarize_binary("PMAP", "Any documented rhythm flag", any_rhythm, f"{pmap_path}"),
    ])

if not mimic.empty and MIMIC_RHYTHM_FEATURES.exists():
    mimic_rhythm = pd.read_csv(MIMIC_RHYTHM_FEATURES)
    mimic_with_rhythm, join_keys = merge_by_available_id(mimic, mimic_rhythm)
    print(f"MIMIC-IV: merged initial rhythm features from {MIMIC_RHYTHM_FEATURES} using keys {join_keys}")

    class_col = find_col(mimic_with_rhythm, ["initial_rhythm_class"])
    shock_col = find_col(mimic_with_rhythm, ["initial_rhythm_shockable"])
    specific_col = find_col(mimic_with_rhythm, ["has_specific_arrest_rhythm_chart_event"])
    if class_col:
        cls = mimic_with_rhythm[class_col].astype(str).str.lower()
        has_specific = cls.isin(["vf", "vt", "pea", "asystole"])
        nonshock = cls.isin(["pea", "asystole"]).where(has_specific, np.nan)
        source = f"{MIMIC_RHYTHM_FEATURES}; initial_rhythm_class; denominator limited to specific VF/VT/PEA/asystole charted rhythm"
    elif shock_col:
        shock = pd.to_numeric(mimic_with_rhythm[shock_col], errors="coerce")
        nonshock = shock.eq(0).where(shock.notna(), np.nan)
        source = f"{MIMIC_RHYTHM_FEATURES}; initial_rhythm_shockable; denominator limited to nonmissing charted rhythm"
    elif specific_col:
        nonshock = pd.Series(np.nan, index=mimic_with_rhythm.index)
        source = f"{MIMIC_RHYTHM_FEATURES}; no class/shockable column found"
    else:
        nonshock = pd.Series(np.nan, index=mimic_with_rhythm.index)
        source = f"{MIMIC_RHYTHM_FEATURES}; no usable initial rhythm column found"

    rhythm_rows.append(summarize_binary(
        "MIMIC-IV",
        "Non-shockable initial rhythm (asystole/PEA)",
        nonshock,
        source=source,
        denominator_note="patients with specific charted VF/VT/PEA/asystole rhythm",
    ))
else:
    rhythm_rows.append({
        "dataset": "MIMIC-IV",
        "variable": "Non-shockable initial rhythm (asystole/PEA)",
        "n": np.nan,
        "denominator": len(mimic) if not mimic.empty else np.nan,
        "percent": np.nan,
        "formatted": "NR",
        "source": f"Missing {MIMIC_RHYTHM_FEATURES}; run mimiciv/identify_initial_rhythm.ipynb first.",
        "denominator_note": "not reported",
    })

rhythm_table = pd.DataFrame(rhythm_rows)
rhythm_component_table = pd.DataFrame(rhythm_components)
display(rhythm_table)
display(rhythm_component_table)
rhythm_table.to_csv(OUT / "table1_nonshockable_rhythm_row.csv", index=False)
rhythm_component_table.to_csv(OUT / "rhythm_component_counts.csv", index=False)

## OHCA/IHCA or ED-Presenting Composition

In [ ]:
location_rows = []
location_detail_rows = []

if not hyperion.empty:
    lieu_col = find_col(hyperion, ["J0_LIEU_ACR"])
    if lieu_col:
        lieu = pd.to_numeric(hyperion[lieu_col], errors="coerce")
        known = lieu.isin([1, 2, 3])
        ohca = lieu.isin([1, 2]).where(known, np.nan)
        ihca = lieu.eq(3).where(known, np.nan)
        location_rows.append(summarize_binary(
            "HYPERION",
            "OHCA",
            ohca,
            source=f"{hyperion_path}; {lieu_col}: 1=home, 2=outside, 3=hospital",
            denominator_note="known J0_LIEU_ACR",
        ))
        location_rows.append(summarize_binary(
            "HYPERION",
            "IHCA",
            ihca,
            source=f"{hyperion_path}; {lieu_col}: 1=home, 2=outside, 3=hospital",
            denominator_note="known J0_LIEU_ACR",
        ))
        counts = hyperion[lieu_col].value_counts(dropna=False).rename_axis("J0_LIEU_ACR").reset_index(name="n")
        counts["dataset"] = "HYPERION"
        counts["label"] = counts["J0_LIEU_ACR"].map({1: "Home", 2: "Outside", 3: "Hospital"})
        location_detail_rows.append(counts)
    else:
        location_rows.append({"dataset": "HYPERION", "variable": "OHCA/IHCA composition", "formatted": "NR", "source": "No J0_LIEU_ACR column found."})

if not eicu.empty:
    source_col = find_col(eicu, ["hospitaladmitsource", "unitadmitsource", "admission_source"])
    if source_col:
        src = eicu[source_col].astype(str).str.strip()
        # Conservative proxy: ED admissions are treated as ED-presenting/OHCA proxy.
        ohca_proxy = src.str.contains(r"emergency|\bed\b", case=False, regex=True, na=False)
        known = src.ne("") & src.str.lower().ne("nan")
        location_rows.append(summarize_binary(
            "eICU-CRD",
            "ED-presenting/OHCA proxy",
            ohca_proxy.where(known, np.nan),
            source=f"{eicu_path}; {source_col}",
            denominator_note="known hospital admit source",
        ))
        location_rows.append(summarize_binary(
            "eICU-CRD",
            "Non-ED/IHCA proxy",
            (~ohca_proxy).where(known, np.nan),
            source=f"{eicu_path}; {source_col}",
            denominator_note="known hospital admit source",
        ))
        counts = src.value_counts(dropna=False).rename_axis("hospital_admit_source").reset_index(name="n")
        counts["dataset"] = "eICU-CRD"
        location_detail_rows.append(counts)
    else:
        location_rows.append({"dataset": "eICU-CRD", "variable": "OHCA/IHCA composition", "formatted": "NR", "source": "No admission source column found."})

if not pmap.empty:
    ed_col = find_col(pmap, ["ed_visit_yn"])
    if ed_col:
        ed = boolish(pmap[ed_col])
        location_rows.append(summarize_binary(
            "PMAP",
            "ED-presenting/OHCA proxy",
            ed,
            source=f"{pmap_path}; {ed_col}",
            denominator_note="analysis cohort",
        ))
        location_rows.append(summarize_binary(
            "PMAP",
            "Non-ED/IHCA proxy",
            ~ed,
            source=f"{pmap_path}; {ed_col}",
            denominator_note="analysis cohort",
        ))
        counts = pmap[ed_col].value_counts(dropna=False).rename_axis("ed_visit_yn").reset_index(name="n")
        counts["dataset"] = "PMAP"
        location_detail_rows.append(counts)
    else:
        location_rows.append({"dataset": "PMAP", "variable": "OHCA/IHCA composition", "formatted": "NR", "source": "No ed_visit_yn column found."})

def load_optional(path):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        print("Loaded", path, df.shape)
        return df
    print("Missing optional file:", path)
    return pd.DataFrame()

ca_ed = load_optional(MIMIC_CA_ED)
ca_time = load_optional(MIMIC_CA_TIME)
if not mimic.empty:
    mimic_ids = mimic[[c for c in ["subject_id", "hadm_id"] if c in mimic.columns]].drop_duplicates()
    if "hadm_id" in mimic_ids.columns and not ca_ed.empty and not ca_time.empty:
        ed_hadm = set(pd.to_numeric(ca_ed.get("hadm_id"), errors="coerce").dropna().astype(int))
        time_hadm = set(pd.to_numeric(ca_time.get("hadm_id"), errors="coerce").dropna().astype(int))
        hadm = pd.to_numeric(mimic_ids["hadm_id"], errors="coerce")
        ed_presenting = hadm.isin(ed_hadm)
        procedure_coded_only = hadm.isin(time_hadm - ed_hadm)
        classified = ed_presenting | procedure_coded_only
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "ED-presenting/OHCA source",
            ed_presenting.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; hadm_id overlap",
            denominator_note="cohort admissions classifiable from CA_ED/CA_time",
        ))
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "Procedure-coded CPR/IHCA source",
            procedure_coded_only.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; hadm_id overlap",
            denominator_note="cohort admissions classifiable from CA_ED/CA_time",
        ))
    elif not ca_ed.empty and not ca_time.empty and "subject_id" in mimic_ids.columns:
        ed_subj = set(pd.to_numeric(ca_ed.get("subject_id"), errors="coerce").dropna().astype(int))
        time_subj = set(pd.to_numeric(ca_time.get("subject_id"), errors="coerce").dropna().astype(int))
        subj = pd.to_numeric(mimic_ids["subject_id"], errors="coerce")
        ed_presenting = subj.isin(ed_subj)
        procedure_coded_only = subj.isin(time_subj - ed_subj)
        classified = ed_presenting | procedure_coded_only
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "ED-presenting/OHCA source",
            ed_presenting.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; subject_id overlap",
            denominator_note="cohort subjects classifiable from CA_ED/CA_time",
        ))
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "Procedure-coded CPR/IHCA source",
            procedure_coded_only.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; subject_id overlap",
            denominator_note="cohort subjects classifiable from CA_ED/CA_time",
        ))
    else:
        location_rows.append({
            "dataset": "MIMIC-IV",
            "variable": "OHCA/IHCA composition",
            "n": np.nan,
            "denominator": len(mimic),
            "percent": np.nan,
            "formatted": "NR",
            "source": "Need CA_ED.csv and CA_time.csv to classify ED-presenting vs procedure-coded source.",
            "denominator_note": "not reported",
        })

location_table = pd.DataFrame(location_rows)
display(location_table)
location_table.to_csv(OUT / "table1_ohca_ihca_composition.csv", index=False)

if location_detail_rows:
    location_detail = pd.concat(location_detail_rows, ignore_index=True, sort=False)
    display(location_detail)
    location_detail.to_csv(OUT / "location_source_value_counts.csv", index=False)

## Copy-Ready Table Rows

In [ ]:
datasets = ["HYPERION", "eICU-CRD", "PMAP", "MIMIC-IV"]

nonshock_map = {}
for _, row in rhythm_table.iterrows():
    nonshock_map[row["dataset"]] = row["formatted"]

# Use the OHCA/ED-presenting row as the primary composition row.
location_map = {}
if "location_table" in globals() and not location_table.empty:
    for dataset in ["HYPERION", "eICU-CRD", "PMAP", "MIMIC-IV"]:
        sub = location_table[(location_table["dataset"] == dataset) & (location_table["variable"].str.contains("OHCA|ED-presenting", case=False, na=False))]
        if len(sub):
            location_map[dataset] = sub.iloc[0]["formatted"]
        else:
            location_map[dataset] = "NR"

copy_ready = pd.DataFrame([
    {"Table 1 row": "Non-shockable initial rhythm (asystole/PEA), n/N (%)", **{d: nonshock_map.get(d, "NR") for d in datasets}},
    {"Table 1 row": "OHCA or ED-presenting arrest, n/N (%)", **{d: location_map.get(d, "NR") for d in datasets}},
])
display(copy_ready)
copy_ready.to_csv(OUT / "copy_ready_table1_rows.csv", index=False)

notes = pd.DataFrame([
    {"note": "MIMIC-IV initial rhythm is read from mimiciv/identify_initial_rhythm.ipynb output when MIMIC_initial_rhythm_features.csv is present, or from MIMIC_INITIAL_RHYTHM_CSV when that environment variable is set."},
    {"note": "HYPERION OHCA uses J0_LIEU_ACR values 1=home and 2=outside; IHCA uses value 3=hospital."},
    {"note": "HYPERION non-shockable rhythm uses J0_RYTHM values 1=asystole and 2=PEA."},
    {"note": "eICU and PMAP OHCA/IHCA rows are admission-source/ED-visit proxies unless independently chart-validated."},
    {"note": "For MIMIC-IV location, CA_ED indicates ED-presenting source and CA_time indicates procedure-coded CPR source; admissions in both files are classified as ED-presenting."},
])
display(notes)
notes.to_csv(OUT / "table1_data_pull_notes.csv", index=False)